# Case Study 1: Zinc5k Pipeline
Run a complete baseline pipeline for LogP prediction on Zinc5k.
Featurize -> Split -> Train -> Evaluate -> Infer

In [ ]:
import uuid
import time
from pathlib import Path 

from pyds import BaseClient, Data, Evaluate, Featurize, Infer, Settings, TVTSplit, Train

In [2]:
BASE_URL = "http://deepchem-server"
PROFILE = "test_profile"
PROJECT = "test_project"

repo_root = Path(".").resolve().parent
DATASET_PATH = repo_root / "deepchem_server" / "core" / "tests" / "assets" / "zinc5k.csv"

run_id = uuid.uuid4().hex[:8]
print(f"Run ID: {run_id} — {time.strftime('%Y-%m-%d %H:%M:%S')}")

Run ID: 585a396a — 2026-03-28 01:03:53


In [3]:
settings = Settings(profile=PROFILE, project=PROJECT, base_url=BASE_URL)

base_client = BaseClient(settings=settings)
data_client = Data(settings=settings)
featurize_client = Featurize(settings=settings)
split_client = TVTSplit(settings=settings)
train_client = Train(settings=settings)
evaluate_client = Evaluate(settings=settings)
infer_client = Infer(settings=settings)

print("healthcheck:", base_client.healthcheck())

healthcheck: {'status': 'ok'}


## 1. Upload dataset
Push the local CSV into the DeepChem datastore for this run.

In [4]:
upload_result = data_client.upload_data(
    file_path=DATASET_PATH,
    filename=f"zinc5k_{run_id}.csv",
    description=f"Zinc5k upload for run {run_id}",
)
dataset_address = upload_result["dataset_address"]
print("dataset_address:", dataset_address)

dataset_address: deepchem://test_profile/test_project/zinc5k_585a396a.csv


## 2. Featurize (ECFP)
Convert SMILES into ECFP fingerprints for model training.

In [5]:
featurize_result = featurize_client.run(
    dataset_address=dataset_address,
    featurizer="ecfp",
    output=f"zinc5k_ecfp_{run_id}",
    dataset_column="smiles",
    label_column="logp",
    feat_kwargs={"radius": 2, "size": 1024},
)
featurized_address = featurize_result["featurized_file_address"]
print("featurized_address:", featurized_address)

featurized_address: deepchem://test_profile/test_project/zinc5k_ecfp_585a396a


## 3. Train / Validation / Test Split
Create train, validation and test subsets for model development.

In [6]:
split_result = split_client.run(
    splitter_type="random",
    dataset_address=featurized_address,
    frac_train=0.8,
    frac_valid=0.1,
    frac_test=0.1,
)
train_addr, valid_addr, test_addr = split_result["train_valid_test_split_results_address"]
print("train:", train_addr)
print("valid:", valid_addr)
print("test: ", test_addr)

train: deepchem://test_profile/test_project/zinc5k_ecfp_585a396a_train
valid: deepchem://test_profile/test_project/zinc5k_ecfp_585a396a_valid
test:  deepchem://test_profile/test_project/zinc5k_ecfp_585a396a_test


## 4. Train (Random Forest Regressor)
Fit a baseline random forest regressor on the training split.

In [7]:
train_result = train_client.run(
    dataset_address=train_addr,
    model_type="random_forest_regressor",
    model_name=f"rf_logp_{run_id}",
    init_kwargs={"n_estimators": 500, "max_features": 0.5, "random_state": 42},
    train_kwargs={},
)
model_address = train_result["trained_model_address"]
print("model_address:", model_address)

model_address: deepchem://test_profile/test_project/rf_logp_585a396a


## 5. Evaluate
Compute evaluation metrics on the test split.

In [8]:
evaluate_result = evaluate_client.run(
    dataset_addresses=[test_addr],
    model_address=model_address,
    metrics=["pearson_r2_score", "rms_score", "mae_error"],
    output_key=f"eval_logp_{run_id}",
    is_metric_plots=False,
)
evaluation_address = evaluate_result["evaluation_result_address"]
print("evaluation_address:", evaluation_address)

evaluation_address: deepchem://test_profile/test_project/eval_logp_585a396a.json


## 6. Infer
Generate predictions on the test split and store them in the datastore.

In [9]:
infer_result = infer_client.run(
    model_address=model_address,
    data_address=test_addr,
    output=f"infer_logp_{run_id}",
    dataset_column="smiles",
)
inference_address = infer_result["inference_results_address"]
print("inference_address:", inference_address)

inference_address: deepchem://test_profile/test_project/infer_logp_585a396a.csv


In [ ]:
eval_results = data_client.get(evaluation_address)
print("Evaluation results:")
print(eval_results)

infer_results = data_client.get(inference_address)
print("\nInference results (preview):")
if hasattr(infer_results, "head"):
    display(infer_results.head())
else:
    print(infer_results)

Evaluation results:
{'deepchem://test_profile/test_project/zinc5k_ecfp_585a396a_test': {'pearson_r2_score': 0.7057955457138442, 'rms_score': 0.8884178412761599, 'mae_score': 0.707053624}}

Inference results (preview):


,X,y_preds
0,CCN1CC[C@H](N2CC(N(C)C(=O)[C@H]3CCN(C)C3)C2)C1=O,-0.068276
1,CC(C)C(=O)NCC[C@@H](C)NC(=O)Cn1cccn1,-0.557752
2,CCC[C@H](CN1C(=O)[C@H]2C[C@@H](O)CN2C1=O)OC,0.269518
3,C=C[C@@H]1OCC[C@H]1C(=O)N[C@@H]1CN(C(=O)COC)C[...,0.173428
4,COc1ccc(OC)c(C=NOCc2cccc3c2OCCO3)c1,2.671914
